# JHU Hub: verify and remove external-redirect rows

The raw scrape (`JHU Hub Articles.csv`) contains three rows without a usable body: one explicit external redirect and two HTTP 403 rows. This notebook re-checks the 403 URLs with Playwright; all three resolve to external (non-Hub) content, so they are removed — the same rule applied to every school: external redirects cannot provide comparable university-produced text.

Output: `./Data/JHU Articles.csv` — the final JHU corpus used by the analysis.


In [ ]:
import pandas as pd

RAW_PATH = "./Data/JHU Hub Articles.csv"
FINAL_PATH = "./Data/JHU Articles.csv"

jhu_df = pd.read_csv(RAW_PATH)
print(f"Raw JHU rows: {len(jhu_df)}")

body_missing = (
    jhu_df["Article Body"].isna()
    | jhu_df["Article Body"].astype("string").str.strip().eq("").fillna(False)
)
problem_rows = jhu_df.loc[body_missing]
print(f"Rows without a usable body: {len(problem_rows)}")
problem_rows[["Article Header", "Url", "External Redirect", "HTTP Status", "Error"]]

## Re-check the HTTP 403 rows with Playwright

Run once; both 403 URLs redirect to external domains.

In [ ]:
import asyncio
from urllib.parse import urlparse

from playwright.async_api import async_playwright

JHU_DOMAIN = "hub.jhu.edu"
recheck_urls = jhu_df.loc[
    pd.to_numeric(jhu_df["HTTP Status"], errors="coerce").eq(403), "Url"
].dropna().tolist()
print(f"URLs to re-check: {recheck_urls}")

async def recheck(urls):
    results = []
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        page = await browser.new_page()
        for url in urls:
            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=45_000)
                final_domain = urlparse(page.url).netloc
                results.append({"Url": url, "Final URL": page.url,
                                "External": JHU_DOMAIN not in final_domain})
            except Exception as exc:
                results.append({"Url": url, "Final URL": "", "External": None,
                                "Error": f"{type(exc).__name__}: {exc}"})
            await asyncio.sleep(2)
        await browser.close()
    return pd.DataFrame(results)

recheck_df = await recheck(recheck_urls)
recheck_df

## Drop the confirmed external rows and save the final corpus

In [ ]:
jhu_final = jhu_df.loc[~body_missing].reset_index(drop=True)
print(f"Rows removed: {len(jhu_df) - len(jhu_final)}")
print(f"Final JHU rows: {len(jhu_final)}")

jhu_final.to_csv(FINAL_PATH, index=False, encoding="utf-8-sig")
print(f"Saved final JHU corpus to: {FINAL_PATH}")